In [ ]:
%pip install -r ../requirements.txt
print('Dependencies installed or already satisfied for Task 1 notebook.')

# Task 1: Exploratory Data Analysis & Data Preprocessing
**CrediTrust Financial — RAG Complaint Chatbot**

This notebook covers:
1. Loading the raw CFPB dataset
2. EDA: product distribution, narrative length, coverage
3. Filtering to the four target product categories
4. Cleaning and normalising complaint narratives
5. Saving the cleaned dataset for downstream tasks

## 1. Setup & Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))   # add project root to path

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocess import (
    load_data,
    map_products,
    filter_narratives,
    clean_text,
    add_word_count,
    TARGET_PRODUCTS,
)

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_colwidth", 80)

print("All imports successful ✅")

## 2. Load Raw Dataset

In [ ]:
RAW_PATH = "../data/raw/complaints.csv"
OUTPUT_PATH = "../data/filtered_complaints.csv"

df_raw = load_data(RAW_PATH)
print(f"Shape: {df_raw.shape}")
df_raw.head(3)

In [ ]:
print("Columns:", df_raw.columns.tolist())
print("\nDtypes:\n", df_raw.dtypes)

## 3. EDA — Raw Dataset

### 3.1 Product Distribution (full dataset)

In [ ]:
product_counts = df_raw["Product"].value_counts()
print(f"Unique products: {df_raw['Product'].nunique()}")
product_counts.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
product_counts.head(15).plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 15 Product Categories — Raw CFPB Dataset", fontsize=14)
ax.set_xlabel("Number of Complaints")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 3.2 Narrative Coverage

In [ ]:
col = "Consumer complaint narrative"
has_narrative = df_raw[col].notna() & (df_raw[col].str.strip() != "")
print(f"Complaints WITH narrative:    {has_narrative.sum():,} ({has_narrative.mean():.1%})")
print(f"Complaints WITHOUT narrative: {(~has_narrative).sum():,} ({(~has_narrative).mean():.1%})")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    [has_narrative.sum(), (~has_narrative).sum()],
    labels=["With Narrative", "Without Narrative"],
    autopct="%1.1f%%",
    colors=["#4CAF50", "#F44336"],
    startangle=140,
)
ax.set_title("Complaint Narrative Coverage")
plt.tight_layout()
plt.show()

## 4. Filter to Target Products

In [ ]:
df_filtered = map_products(df_raw)
print(f"After product filter: {len(df_filtered):,} rows")
df_filtered["product_category"].value_counts()

In [ ]:
df_filtered = filter_narratives(df_filtered)
print(f"After narrative filter: {len(df_filtered):,} rows")

### 4.1 Product Distribution After Filtering

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
counts = df_filtered["product_category"].value_counts()
sns.barplot(x=counts.index, y=counts.values, palette="viridis", ax=ax)
ax.set_title("Complaint Distribution by Target Product Category", fontsize=13)
ax.set_xlabel("Product Category")
ax.set_ylabel("Number of Complaints")
ax.bar_label(ax.containers[0], fmt="%d", padding=3)
plt.tight_layout()
plt.show()

## 5. Text Cleaning

In [ ]:
df_filtered = df_filtered.copy()
df_filtered["cleaned_narrative"] = df_filtered["Consumer complaint narrative"].apply(clean_text)

before = len(df_filtered)
df_filtered = df_filtered[df_filtered["cleaned_narrative"].str.len() > 20].reset_index(drop=True)
print(f"Post-clean filter: {before} → {len(df_filtered)} rows")

In [ ]:
sample = df_filtered[["Consumer complaint narrative", "cleaned_narrative"]].sample(3, random_state=42)
for i, row in sample.iterrows():
    print("─" * 70)
    print("ORIGINAL:", row["Consumer complaint narrative"][:300])
    print()
    print("CLEANED: ", row["cleaned_narrative"][:300])
    print()

## 6. Narrative Length Analysis

In [ ]:
df_filtered = add_word_count(df_filtered)

print("Word count statistics:")
print(df_filtered["word_count"].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df_filtered["word_count"].clip(upper=500).hist(bins=50, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Word Count Distribution (capped at 500 words)")
axes[0].set_xlabel("Word Count")
axes[0].set_ylabel("Frequency")

sns.boxplot(
    data=df_filtered,
    x="product_category",
    y="word_count",
    palette="Set2",
    showfliers=False,
    ax=axes[1],
)
axes[1].set_title("Word Count by Product Category")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
print(f"Very short (<20 words):  {(df_filtered['word_count'] < 20).sum():,}")
print(f"Very long  (>500 words): {(df_filtered['word_count'] > 500).sum():,}")
print(f"Median word count:       {df_filtered['word_count'].median():.0f}")

## 7. Summary Statistics

In [ ]:
print("=" * 55)
print("        EDA SUMMARY — CrediTrust RAG Project")
print("=" * 55)
print(f"  Total complaints (filtered & cleaned): {len(df_filtered):,}")
print()
print("  Product distribution:")
for cat, cnt in df_filtered["product_category"].value_counts().items():
    print(f"    {cat:<22} {cnt:>7,}  ({cnt/len(df_filtered):.1%})")
print()
print(f"  Narrative word count")
print(f"    Mean   : {df_filtered['word_count'].mean():.0f}")
print(f"    Median : {df_filtered['word_count'].median():.0f}")
print(f"    Min    : {df_filtered['word_count'].min()}")
print(f"    Max    : {df_filtered['word_count'].max()}")
print("=" * 55)

## 8. Save Cleaned Dataset

In [ ]:
output_cols = {
    "Complaint ID": "complaint_id",
    "Product": "product_raw",
    "product_category": "product_category",
    "Issue": "issue",
    "Sub-issue": "sub_issue",
    "Consumer complaint narrative": "original_narrative",
    "cleaned_narrative": "cleaned_narrative",
    "word_count": "word_count",
    "Company": "company",
    "State": "state",
    "Date received": "date_received",
}
existing = {k: v for k, v in output_cols.items() if k in df_filtered.columns}
df_out = df_filtered.rename(columns=existing)[list(existing.values())]

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_out.to_csv(OUTPUT_PATH, index=False)
print(f"✅  Saved {len(df_out):,} rows → {OUTPUT_PATH}")
df_out.head(3)